# AuroraBid: End-to-End Pipeline Demonstration
This notebook demonstrates the full simulation workflow of the AuroraBid engine. It includes feature loading, bidding strategy execution, RL policy application, auction result simulation, and final evaluation.

In [1]:
# 1) 切到项目根（换成你的实际路径）
%cd /Users/shawn/Documents/MLLESSONS/AuroraBid

# 2) 让解释器能找到 src 下的包
import sys
sys.path.insert(0, "src")

# 3) 正确导入
from features.feature_store import LocalFeatureStore

# 4) 读数据用项目相对路径
from pathlib import Path
DATA_DIR = Path("data/processed")


/Users/shawn/Documents/MLLESSONS/AuroraBid


## Step 1: Load Processed Feature Data
We start by loading the processed user/context/history/joined features from the `data/processed` folder.

In [2]:
import pandas as pd
from pathlib import Path

# Define path to processed feature files

# Load all feature sets
user_df = pd.read_csv(r"/Users/shawn/Documents/MLLESSONS/AuroraBid/data/processed/user_embeddings.csv")
context_df = pd.read_csv(r"/Users/shawn/Documents/MLLESSONS/AuroraBid/data/processed/context_features.csv")
history_df = pd.read_csv(r"/Users/shawn/Documents/MLLESSONS/AuroraBid/data/processed/history_features.csv")

# Display the first few rows of the joined feature table
print("用户特征:")
print(user_df.head())
print("\n上下文特征:")
print(context_df.head())
print("\n历史特征:")
print(history_df.head())

用户特征:
  user_id  user_emb_1  user_emb_2  user_emb_3
0   u9286   -1.151786   -1.129462   -0.427612
1   u8338    2.129885    0.537058   -0.418567
2   u7162    0.882593   -0.693025    1.286422
3   u9961    0.151564   -1.361205   -0.443207
4   u4874    0.012090   -0.722436    0.895674

上下文特征:
  adslot_id  adslot_enc  media_gaming  media_newsapp  media_social  \
0       s01           0           0.0            1.0           0.0   
1       s02           1           0.0            0.0           0.0   
2       s03           2           0.0            0.0           1.0   
3       s04           3           1.0            0.0           0.0   

   media_video  device_mobile  device_pc  os_android  os_ios  os_mac  os_win  \
0          0.0            1.0        0.0         0.0     1.0     0.0     0.0   
1          1.0            0.0        1.0         0.0     0.0     0.0     1.0   
2          0.0            1.0        0.0         1.0     0.0     0.0     0.0   
3          0.0            0.0        1.

## Step 2: Initialize Feature Store Interface
The `LocalFeatureStore` provides access to features as if they're from Redis/Kafka. This is a mock used in offline training and testing.

In [ ]:
from src.features.feature_store import LocalFeatureStore

# Initialize and load mock feature store
fs = LocalFeatureStore('data/processed')
fs.load()

# Example: Retrieve features for a specific request
combined_df = fs.get_combined_feature(user_id='user_001', adslot_id='ad_001', request_id='req_001')

In [4]:
import sys, platform
print("Python 可执行文件：", sys.executable)
print("Python 版本：", sys.version)

Python 可执行文件： /Users/shawn/miniforge3/envs/aurorabid/bin/python
Python 版本： 3.10.18 (main, Jun  5 2025, 08:37:47) [Clang 14.0.6 ]


In [5]:
import sys
!"{sys.executable}" -m pip show torch


Name: torch
Version: 2.0.1
Summary: Tensors and Dynamic neural networks in Python with strong GPU acceleration
Home-page: https://pytorch.org/
Author: PyTorch Team
Author-email: packages@pytorch.org
License: BSD-3
Location: /Users/shawn/miniforge3/envs/aurorabid/lib/python3.10/site-packages
Requires: typing-extensions, filelock, networkx, sympy, jinja2
Required-by: torchvision, stable-baselines3


## Step 3: Load Models & Bidding Strategy
We load the trained RL models or rules from disk using `ModelServer`, and wrap them into a unified `BidStrategyWrapper`.

In [6]:
from src.serving.model_server import ModelServer
from src.bidding.bid_strategy import BidStrategyWrapper

# Initialize model server and load models
ms = ModelServer(checkpoint_dir='checkpoints')
strategy = BidStrategyWrapper(strategy_type='rl', model_dict=ms.get_all())

## Step 4: Generate Bids Using Strategy
We loop over each impression record and apply our bid strategy (e.g. RL agent or rule-based logic) to generate a bid price.

In [7]:
sample_bids = []

# For each row, generate a bid price using selected strategy
for _, row in combined_df.iterrows():
    bid = strategy.bid(row)
    sample_bids.append(bid)

# Append bid results to dataframe
combined_df['predicted_bid'] = sample_bids
combined_df[['request_id', 'predicted_bid']].head()

NameError: name 'combined_df' is not defined

## Step 5: Simulate Auction Results
We use a `BidLandscapeSimulator` to generate competitors’ bids and then resolve auction results using second-price logic.

In [ ]:
from src.simulation.bid_simulator import BidLandscapeSimulator
from src.simulation.auction_engine import SecondPriceAuction

bls = BidLandscapeSimulator()
auction = SecondPriceAuction()

# Run simulated auction
results = []
for _, row in combined_df.iterrows():
    competitor_bids = bls.sample()
    win, price = auction.resolve(row['predicted_bid'], competitor_bids)
    results.append({'win': win, 'price_paid': price})

# Merge with main dataframe
auction_df = pd.DataFrame(results)
full_df = pd.concat([combined_df.reset_index(drop=True), auction_df], axis=1)
full_df.head()

## Step 6: Evaluate Performance (CTR / CVR / ROI)
Now that we have bids and auction results, we can evaluate the effectiveness of our strategy using common metrics.

In [ ]:
from src.utils.metrics import compute_roi, compute_ctr, compute_cvr

# Evaluate key metrics
roi = compute_roi(full_df['click'], full_df['price_paid'], full_df['value'])
ctr = compute_ctr(full_df['click'])
cvr = compute_cvr(full_df['click'], full_df['conversion'])

print(f"ROI: {roi:.4f} | CTR: {ctr:.4f} | CVR: {cvr:.4f}")

## Step 7: Visualize Bid Distributions & Auction Behavior

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

# Bid distribution
sns.histplot(full_df['predicted_bid'], bins=40, kde=True)
plt.title('Predicted Bid Distribution')
plt.xlabel('Bid Price')
plt.ylabel('Count')
plt.show()

# Auction win-rate vs bid
sns.scatterplot(data=full_df, x='predicted_bid', y='win', alpha=0.3)
plt.title('Bid vs Auction Win')
plt.xlabel('Predicted Bid')
plt.ylabel('Win (1=True)')
plt.show()